In [ ]:
import safetensors.torch
from models.FEMBA import FEMBA, FembaEncoder

In [ ]:
from huggingface_hub import snapshot_download

# downloads all task folders (TUAB/TUAR/TUSL) and safetensors into ./checkpoints/FEMBA
snapshot_download(repo_id="thorir/FEMBA", repo_type="model", local_dir="checkpoints/FEMBA")

In [ ]:
def load_model_from_safetensors(safetensors_path, device="cpu"):
    weights = safetensors.torch.load_file(safetensors_path)
    model_full = FEMBA(num_classes=2)
    model_encoder = FembaEncoder()
    for model in model_full, model_encoder:
        model.load_state_dict(weights, strict=False)
        model.eval()
        model.to(device)
    
    return model_full, model_encoder

device="cuda"
model_f, model_e = load_model_from_safetensors("checkpoints/FEMBA/TUAR/FEMBA_large.safetensors", device)

In [ ]:
import torch

batch_size = 4
seq_length = 1280
num_channels = 22

# Create random EEG data (batch_size, channels, seq_length)
x = torch.randn(batch_size, num_channels, seq_length).to(device)

# Create a mask (same shape as input, boolean)
# Mask indicates which positions should be masked (set to True for masked positions)
mask = torch.zeros(batch_size, num_channels, seq_length, dtype=torch.bool).to(device)
# Mask some random positions (e.g., mask 10% of the input)
mask[:, :, torch.randint(0, seq_length, (int(0.1 * seq_length),))] = True

# Forward pass
with torch.no_grad():  # Disable gradient computation for inference
    output, original = model_f(x, mask)
    embedding = model_e(x, mask)

print(f"Input shape: {x.shape}")
print(f"Mask shape: {mask.shape}")
print(f"Output shape: {output.shape}")  # For classification: (batch_size, num_classes)
print(f"Embedding shape: {embedding.shape}")
print(f"Original shape: {original.shape}")  # (batch_size, num_channels, seq_length)